# SUSY_DRU · 7 vs 8 variables con PennyLane

Este notebook compara el clasificador de **un qubit con *data re-uploading*** con dos conjuntos de variables
(las **7 de la corrida final** y las **8 primarias** de SUSY) y tres funciones de pérdida. Parte de la misma base que
`colab/SUSY_DRU_PennyLane_colab.ipynb`: mismos datos, mismo split, mismo escalado por cuantiles, mismo circuito de
5 capas y misma receta de entrenamiento que la corrida final del 7/11/2026.

| Paso | Qué hace | Modo rápido | Modo completo |
|---|---|---|---|
| 0 | Configuración: modo, nombre del experimento, carpeta de Drive | — | — |
| 1 | Clona el repo, instala PennyLane 0.45.1 y monta Drive | 1–2 min | 1–2 min |
| 2 | Baja SUSY y verifica el split firmado (5000 / 1000 / 5000) | < 1 min | < 1 min |
| 3 | Define el experimento (variables, pérdidas, búsqueda) y prepara los datos | segundos | segundos |
| 4 | Define el circuito en PennyLane y lo verifica | < 1 min | < 1 min |
| 5 | Entrena todas las corridas y guarda cada una en Drive al terminarla | 2–4 min | ~3 h |
| 6 | Elige el mejor ajuste de cada grupo **solo con validación** | segundos | segundos |
| 7 | Evalúa en test los modelos elegidos | < 1 min | < 1 min |
| 8 | Gráficas | < 1 min | < 1 min |
| 9 | Resumen de archivos y equivalencias con la corrida final | — | — |

### El experimento

Seis grupos: 2 conjuntos de variables × 3 pérdidas.

| Grupo | Pérdida | Lectura del qubit |
|---|---|---|
| 7 u 8 variables · BCE | entropía cruzada binaria | ⟨X⟩, ⟨Y⟩, ⟨Z⟩ → sigmoide con 4 parámetros entrenables |
| 7 u 8 variables · infidelidad lineal | ⟨1 − F⟩ | solo ⟨Z⟩: p(señal) = (1 − ⟨Z⟩)/2 |
| 7 u 8 variables · infidelidad cuadrática | ⟨(1 − F)²⟩ | solo ⟨Z⟩: p(señal) = (1 − ⟨Z⟩)/2 |

con F = (1 + s⟨Z⟩)/2, s = +1 para fondo y −1 para señal (las mismas pérdidas que `dru_core`).

- **7 variables:** lep1_pt, lep2_pt, met, m_tr_2, m_delta_r, lep1_eta, lep2_eta (las de la corrida final).
- **8 variables primarias** (las de bajo nivel de SUSY): lep1_pt, lep2_pt, met, lep1_eta, lep2_eta, lep1_phi, lep2_phi, met_phi.
- **Circuito por capa:** Rot · Rot · RY con 7 variables, y Rot · Rot · RY · **RZ** con 8. Cada ángulo es w·x + b.
- **Búsqueda:** tasa η ∈ {0.005, 0.01, 0.02} × lote {64, 128} × semillas {101, 202, 303} = 18 corridas por grupo,
  **108 en total**. η escala la receta de la corrida final: capa por capa a 5η, fase 1 (12 épocas) a 2η y fase 2
  (18 épocas) a η. Con η = 0.01 y lote 128 es **exactamente** la receta de la corrida final.
- **Selección:** en cada grupo gana el ajuste con mayor AUC de validación medio en las 3 semillas (empate exacto:
  menor η y luego menor lote). La selección se congela en Drive **antes** de mirar test.
- **Test:** se usa una sola vez, al final, con los 18 modelos elegidos (6 grupos × 3 semillas).

### Dos modos

- `MODO = "rapido"`: solo comprueba que todo corre de principio a fin (1200 eventos de train, 400 de validación,
  2 + 1 épocas en las fases, 2 tasas, 1 lote, 2 semillas: 24 corridas). **Sus números no significan nada.**
  Se guarda en una carpeta aparte (`<EXPERIMENTO>_rapido`) para no mezclarse con el experimento.
- `MODO = "completo"`: el experimento de verdad.

### Antes de empezar (una vez)

1. Nada que configurar para clonar: el repo es público. (Solo con una copia privada hace falta el Secret `GH_TOKEN`.)
2. Entorno de ejecución: **CPU**. Con 1 qubit la GPU no acelera nada.
3. En el paso 1 Colab pide permiso para acceder a tu Drive: acéptalo. Ahí se guarda todo.

**Si Colab se desconecta:** vuelve a ejecutar todo con el mismo `EXPERIMENTO`. Las corridas que ya están en Drive se
saltan y se sigue con la siguiente; solo se pierde la que estaba a medias. Deja la pestaña abierta mientras corre.

**Sobre el test:** el test oficial ya se usó una vez para la corrida final pre-registrada. Lo que salga aquí es
exploratorio: sirve para comparar 7 con 8 variables, pero no reemplaza los resultados de la corrida final.

## 0 · Configuración

Es la única celda que hay que tocar. Primero córrelo todo en `"rapido"`; si llega al final sin errores, cambia a
`"completo"` y vuelve a ejecutar todo.

In [ ]:
MODO = "completo"                     # "rapido": comprueba que todo corre · "completo": el experimento de verdad
EXPERIMENTO = "7v8_pennylane_v1"    # nombre de la carpeta de resultados; usa otro nombre para empezar de cero
GUARDAR_EN_DRIVE = True             # False: se guarda en /content/susy-dru/colab/salidas y se borra al cerrar Colab
CARPETA_DRIVE = "SUSY_DRU"          # carpeta dentro de "Mi unidad"

assert MODO in ("rapido", "completo"), 'MODO tiene que ser "rapido" o "completo"'

## 1 · Repo, dependencias y Drive

Igual que en el notebook de PennyLane, con la versión fijada en 0.45.1 (la que ya reprodujo la corrida final).

In [ ]:
import os, sys, subprocess, pathlib

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "github.com/renzovega-creator/susy-dru.git"
VERSION_PENNYLANE = "0.45.1"

if IN_COLAB:
    from google.colab import userdata
    REPO = pathlib.Path("/content/susy-dru")
    if not REPO.exists():
        # El repo es público: el token solo hace falta si se trabaja con una copia privada (Secret GH_TOKEN).
        try:
            token = userdata.get("GH_TOKEN")
        except Exception:
            token = None
        url = f"https://{token}@{REPO_URL}" if token else f"https://{REPO_URL}"
        r = subprocess.run(["git", "clone", "--depth", "1", url, str(REPO)], capture_output=True, text=True)
        if r.returncode:
            raise RuntimeError("git clone falló: " + (r.stderr.replace(token, "***") if token else r.stderr))
        if token:
            # que el token no quede guardado en .git/config
            subprocess.run(["git", "-C", str(REPO), "remote", "set-url", "origin", f"https://{REPO_URL}"], check=True)
        del token
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"pennylane=={VERSION_PENNYLANE}"], check=True)
else:
    # Ejecución local: el notebook vive en <repo>/colab/
    REPO = pathlib.Path(os.environ.get("SUSY_DRU_REPO", pathlib.Path.cwd().parent)).resolve()

sys.path[:0] = [str(REPO / "corrida_final"), str(REPO / "replica_dru")]
COMMIT = subprocess.run(["git", "-C", str(REPO), "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip()
print("repo:", REPO, "·", COMMIT)

# Carpeta de resultados: todo lo que produce el notebook va aquí
if IN_COLAB and GUARDAR_EN_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = pathlib.Path("/content/drive/MyDrive") / CARPETA_DRIVE
else:
    BASE = REPO / "colab" / "salidas"
SALIDA = BASE / (EXPERIMENTO + ("_rapido" if MODO == "rapido" else ""))
for sub in ("corridas", "test", "tablas", "figuras"):
    (SALIDA / sub).mkdir(parents=True, exist_ok=True)
print("resultados en:", SALIDA)

repo: /content/susy-dru · 6d8f93a Añade la corrida final del 7/11 y un notebook de Colab con PennyLane
Mounted at /content/drive
resultados en: /content/drive/MyDrive/SUSY_DRU/7v8_pennylane_v1


## 2 · Datos

Los mismos que la corrida final: las primeras 26 000 filas de `SUSY.csv.gz`, con el split firmado
(5000 train / 1000 validación / 5000 test, balanceados). Si el CSV o el split no coinciden, la celda se detiene.

In [ ]:
import hashlib
import numpy as np
import pandas as pd
import susy_data
import corrida_final as cf

f60 = REPO / "replica_dru" / "data" / "susy_head_60000.csv"
if not f60.exists():
    susy_data.load_susy(n_rows=60_000)          # descarga y guarda en replica_dru/data/
sha = hashlib.sha256(f60.read_bytes().replace(b"\r\n", b"\n")).hexdigest()
assert sha == "f10ddc24857f874eacc37c3997db3450119032bb9c5a45cbcec29faf7eb721b6", f"CSV distinto al de la corrida: {sha}"

cf.load_data()     # lee las 26 000 filas, regenera el split y lo compara con split_comun.npz (assert)
print({k: len(v) for k, v in cf.IDX.items()}, "· señal en test:", int(cf.Y[cf.IDX["test"]].sum()))

{'train': 5000, 'val': 1000, 'test': 5000} · señal en test: 2500


## 3 · Diseño del experimento

Todo sale de la receta de la corrida final (`HP_BASE`); cada corrida cambia solo variables, pérdida, lectura, tasa y
lote. El `assert` comprueba que la búsqueda contiene la receta exacta de la corrida final.

El preprocesamiento es el del repo (`cf.prepare`: escalado por cuantiles a [−π, π], ajustado solo con train). El test
es el mismo para todos los grupos, así que la comparación entre 7 y 8 variables es pareada (mismos eventos).

In [ ]:
import itertools, json
from dataclasses import replace

HP_BASE = cf.CONFIGS["M2"]       # receta de la corrida final con 7 variables y BCE (equivalencias en el paso 9)
N_CAPAS = HP_BASE.n_layers        # 5
ETA_BASE = HP_BASE.stages[-1][1]  # 0.01: tasa de la última fase

VARIABLES = {
    7: tuple(HP_BASE.features),
    8: ("lep1_pt", "lep2_pt", "met", "lep1_eta", "lep2_eta", "lep1_phi", "lep2_phi", "met_phi"),
}
# nombre en este notebook -> (pérdida en dru_core, lectura, nombre para tablas y gráficas)
PERDIDAS = {
    "bce":                    ("bce",        "xyz", "BCE"),
    "infidelidad_lineal":     ("infidelity", "z",   "infidelidad lineal"),
    "infidelidad_cuadratica": ("fidelity",   "z",   "infidelidad cuadrática"),
}
GRUPOS = [(n, l) for n in VARIABLES for l in PERDIDAS]


def nombre(grupo):
    n, l = grupo
    return f"{n} variables · {PERDIDAS[l][2]}"


if MODO == "completo":
    ETAS, LOTES, SEMILLAS = (0.005, 0.01, 0.02), (64, 128), (101, 202, 303)
    N_TRAIN, N_VAL, EPOCAS_FASE = None, None, None      # todo el split y la receta completa
else:
    ETAS, LOTES, SEMILLAS = (0.01, 0.02), (128,), (101, 202)
    N_TRAIN, N_VAL, EPOCAS_FASE = 1200, 400, (2, 1)     # recortes del modo rápido


def receta(n_vars, perdida, eta, lote):
    """Receta de la corrida final con otras variables, pérdida, tasa y lote. η escala todas las tasas
    (capa por capa a 5η, fase 1 a 2η, fase 2 a η); con η = 0.01 son exactamente las de la corrida final."""
    loss, lectura, _ = PERDIDAS[perdida]
    f = eta / ETA_BASE
    return replace(HP_BASE, features=VARIABLES[n_vars], loss=loss, readout=lectura, batch_size=lote,
                   layerwise_lr=round(HP_BASE.layerwise_lr * f, 12),
                   stages=tuple((n_ep, round(lr * f, 12)) for n_ep, lr in HP_BASE.stages))


def hp_de(c):
    hp = receta(c["n_vars"], c["perdida"], c["eta"], c["lote"])
    if EPOCAS_FASE:                                     # modo rápido: menos épocas por fase
        hp = replace(hp, stages=tuple((n_ep, lr) for n_ep, (_, lr) in zip(EPOCAS_FASE, hp.stages)))
    return hp


assert receta(7, "bce", ETA_BASE, HP_BASE.batch_size) == HP_BASE, "la búsqueda no contiene la receta de la corrida final"


def run_id(c):
    return f"v{c['n_vars']}_{c['perdida']}_eta{c['eta']:g}_b{c['lote']}_s{c['semilla']}"


def es_ref(c):
    """La receta de la corrida final: 7 variables, BCE, η = 0.01, lote 128."""
    return (c["n_vars"], c["perdida"], c["eta"], c["lote"]) == (7, "bce", ETA_BASE, HP_BASE.batch_size)


CORRIDAS = [dict(n_vars=n, perdida=l, eta=e, lote=b, semilla=s)
            for (n, l), e, b, s in itertools.product(GRUPOS, ETAS, LOTES, SEMILLAS)]
CORRIDAS.sort(key=lambda c: not es_ref(c))          # primero la receta de la corrida final; el resto, en orden

# ── Datos de cada grupo ──────────────────────────────────────────────────────
faltan = sorted({v for vs in VARIABLES.values() for v in vs} - set(cf.COLS))
assert not faltan, f"variables que no están en SUSY: {faltan}"


def recortar(X, y, n):
    """Modo rápido: los primeros n/2 eventos de cada clase (siempre los mismos)."""
    idx = np.sort(np.concatenate([np.flatnonzero(y == c)[: n // 2] for c in (0, 1)]))
    return X[idx], y[idx]


DATOS, HUELLA = {}, {}
for g in GRUPOS:
    d = cf.prepare(receta(*g, ETA_BASE, HP_BASE.batch_size))
    if N_TRAIN:
        d["Xtr"], d["ytr"] = recortar(d["Xtr"], d["ytr"], N_TRAIN)
        d["Xva"], d["yva"] = recortar(d["Xva"], d["yva"], N_VAL)
    assert d["use_val"] and d["Xtr"].shape[1] == len(VARIABLES[g[0]])
    DATOS[g] = d
    h = hashlib.sha256()                                # huella de train + val: detecta si cambian los datos
    for k in ("Xtr", "ytr", "Xva", "yva"):
        h.update(np.ascontiguousarray(d[k], dtype=float).tobytes())
    HUELLA[g] = h.hexdigest()[:16]
YTE = DATOS[GRUPOS[0]]["yte"]
assert all(np.array_equal(DATOS[g]["yte"], YTE) for g in GRUPOS), "el test tiene que ser el mismo en todos los grupos"

print(f"Modo {MODO}: {len(GRUPOS)} grupos × {len(ETAS)} tasas × {len(LOTES)} lotes × {len(SEMILLAS)} semillas"
      f" = {len(CORRIDAS)} corridas")
display(pd.DataFrame([dict(grupo=nombre(g), lectura=PERDIDAS[g[1]][1].upper(), variables=", ".join(VARIABLES[g[0]]),
                           train=len(DATOS[g]["ytr"]), val=len(DATOS[g]["yva"]), test=len(DATOS[g]["yte"]),
                           rango_angulos=f"[{DATOS[g]['Xtr'].min():.2f}, {DATOS[g]['Xtr'].max():.2f}]")
                      for g in GRUPOS]))

Modo completo: 6 grupos × 3 tasas × 2 lotes × 3 semillas = 108 corridas


,grupo,lectura,variables,train,val,test,rango_angulos
0,7 variables · BCE,XYZ,"lep1_pt, lep2_pt, met, m_tr_2, m_delta_r, lep1...",5000,1000,5000,"[-3.14, 3.14]"
1,7 variables · infidelidad lineal,Z,"lep1_pt, lep2_pt, met, m_tr_2, m_delta_r, lep1...",5000,1000,5000,"[-3.14, 3.14]"
2,7 variables · infidelidad cuadrática,Z,"lep1_pt, lep2_pt, met, m_tr_2, m_delta_r, lep1...",5000,1000,5000,"[-3.14, 3.14]"
3,8 variables · BCE,XYZ,"lep1_pt, lep2_pt, met, lep1_eta, lep2_eta, lep...",5000,1000,5000,"[-3.14, 3.14]"
4,8 variables · infidelidad lineal,Z,"lep1_pt, lep2_pt, met, lep1_eta, lep2_eta, lep...",5000,1000,5000,"[-3.14, 3.14]"
5,8 variables · infidelidad cuadrática,Z,"lep1_pt, lep2_pt, met, lep1_eta, lep2_eta, lep...",5000,1000,5000,"[-3.14, 3.14]"


## 4 · El circuito en PennyLane

`ModeloPL` es el del notebook de PennyLane, ampliado para 8 variables. Por capa aplica
Rot(x₀, x₁, x₂) · Rot(x₃, x₄, x₅) · RY(x₆) y, con 8 variables, además RZ(x₇); cada ángulo es w·x + b, con peso y
sesgo propios por capa y variable.

**Parámetros.** Con 7 variables se usa el mapa de parámetros del repo tal cual (`dru_core.build_circuit("muten", 7, ...)`),
la misma inicialización (`cf.init_params`) y las mismas máscaras capa por capa (`dru_core.layer_mask`). Con 8 variables
se añade al final del vector un peso y un sesgo por capa para la octava variable; arrancan con la misma distribución
que el resto (se toman de la propia `cf.init_params` con otra semilla) y entran en la máscara de su capa.

| Grupo | Parámetros |
|---|---|
| 7 variables · BCE | 70 de compuertas + 4 de lectura = 74 |
| 7 variables · infidelidades | 70 |
| 8 variables · BCE | 80 + 4 = 84 |
| 8 variables · infidelidades | 80 |

**Verificación** (con parámetros genéricos y 64 eventos de train):
- 7 variables: contra el motor NumPy del repo (`dru_core.DRUModel`), como en el notebook de PennyLane.
- 8 variables: contra una simulación independiente con matrices 2×2 (sin PennyLane) y gradientes por diferencias finitas.
- Con la RZ extra apagada (w = b = 0), el circuito de 8 variables tiene que dar lo mismo que el de 7.

In [ ]:
import pennylane as qml
from pennylane import numpy as pnp
from dru_core import DRUModel, build_circuit, Adam, layer_mask

print("PennyLane", qml.__version__)
if qml.__version__ != VERSION_PENNYLANE:
    print(f"Aviso: este notebook se comprobó con PennyLane {VERSION_PENNYLANE}")


class Arquitectura:
    """Dónde está cada parámetro. 7 variables: el mapa del repo tal cual. 8 variables: el mismo, más un peso y
    un sesgo por capa para la octava variable al final del vector."""

    def __init__(self, n_vars, readout):
        assert n_vars in (7, 8)
        self.n_vars, self.readout = n_vars, readout
        self.circ7 = c = build_circuit("muten", 7, N_CAPAS, 1, readout)
        assert c.n_wires == 1
        L = self.n_capas = c.n_layers
        self.iw = np.full((L, n_vars), -1)
        self.ib = np.full((L, n_vars), -1)
        for k, g in enumerate(c.gates):
            self.iw[c.layer_of_gate[k], g.feat] = g.w_idx
            self.ib[c.layer_of_gate[k], g.feat] = g.b_idx
        self.n_params = c.n_params
        if n_vars == 8:
            extra = c.n_params + 2 * np.arange(L)
            self.iw[:, 7], self.ib[:, 7] = extra, extra + 1
            self.n_params += 2 * L
        self.readout_slice = c.readout_slice
        idx = np.concatenate([self.iw.ravel(), self.ib.ravel()])
        lectura = np.arange(c.n_params)[c.readout_slice] if readout == "xyz" else np.array([], int)
        assert (idx >= 0).all(), "a alguna compuerta le falta su peso o su sesgo"
        assert len(set(idx)) == len(idx) and not set(idx) & set(lectura), "parámetros repetidos"

    def init(self, hp, semilla):
        if self.n_vars == 7:
            return np.asarray(cf.init_params(hp, self.circ7, semilla), float)
        hp7 = replace(hp, features=VARIABLES[7])
        p = np.asarray(cf.init_params(hp7, self.circ7, semilla), float)
        aux = np.asarray(cf.init_params(hp7, self.circ7, 10_000 + semilla), float)   # misma distribución que el resto
        extra = np.empty(2 * self.n_capas)
        extra[0::2], extra[1::2] = aux[self.iw[:, 6]], aux[self.ib[:, 6]]
        return np.concatenate([p, extra])

    def mascara(self, capa):
        """Máscara del repo; con 8 variables, la octava entra en cada máscara igual que la séptima de su misma capa."""
        m = np.asarray(layer_mask(self.circ7, capa))
        if self.n_vars == 7:
            return m
        if m.shape == (self.circ7.n_params,):              # máscara 0/1 (o booleana) sobre todos los parámetros
            extra = np.empty(2 * self.n_capas, dtype=m.dtype)
            extra[0::2], extra[1::2] = m[self.iw[:, 6]], m[self.ib[:, 6]]
            return np.concatenate([m, extra])
        dentro = set(m.tolist())                           # máscara como lista de índices
        extra = [i for j in range(self.n_capas) if self.iw[j, 6] in dentro for i in (self.iw[j, 7], self.ib[j, 7])]
        return np.concatenate([m, np.array(extra, dtype=m.dtype)])


class ModeloPL:
    """Misma interfaz que dru_core.DRUModel (predict_proba, loss_and_grad); el estado lo simula PennyLane."""

    def __init__(self, arq, loss="bce", alpha=0.5, device="default.qubit"):
        self.arq, self.loss, self.alpha = arq, loss, alpha
        L, d = arq.n_capas, arq.n_vars
        dev = qml.device(device, wires=1)

        @qml.qnode(dev, interface="autograd", diff_method="backprop")
        def bloch(X, w, b):                      # X: (B, d); w, b: (L, d). Broadcasting sobre el lote.
            for l in range(L):
                a = w[l] * X + b[l]
                qml.Rot(a[:, 0], a[:, 1], a[:, 2], wires=0)
                qml.Rot(a[:, 3], a[:, 4], a[:, 5], wires=0)
                qml.RY(a[:, 6], wires=0)
                if d == 8:
                    qml.RZ(a[:, 7], wires=0)
            return qml.expval(qml.PauliX(0)), qml.expval(qml.PauliY(0)), qml.expval(qml.PauliZ(0))

        self.bloch = bloch

    def _prob_y_z(self, p, X):
        ex, ey, ez = self.bloch(X, p[self.arq.iw], p[self.arq.ib])
        if self.arq.readout == "xyz":
            r = self.arq.readout_slice.start
            u = p[r] * ex + p[r + 1] * ey + p[r + 2] * ez + p[r + 3]
            return 0.5 * (1.0 + pnp.tanh(0.5 * u)), ez
        return 0.5 * (1.0 - ez), ez

    def _loss(self, p, X, y):
        prob, ez = self._prob_y_z(p, X)
        F = 0.5 * (1.0 + (1.0 - 2.0 * y) * ez)
        if self.loss == "bce":
            return pnp.mean(-(y * pnp.log(prob + 1e-9) + (1 - y) * pnp.log(1 - prob + 1e-9)))
        if self.loss == "infidelity":
            return pnp.mean(1.0 - F)
        if self.loss == "fidelity":
            return pnp.mean((1.0 - F) ** 2)
        raise ValueError(self.loss)

    def loss_and_grad(self, p, X, y):
        pp = pnp.array(p, requires_grad=True)
        Xc, yc = np.asarray(X, float), np.asarray(y, float)   # datos: arrays NumPy simples (no se derivan)
        grad_fn = qml.grad(self._loss, argnums=0)
        g = grad_fn(pp, Xc, yc)
        return float(grad_fn.forward), np.asarray(g, float)   # .forward = valor de la pérdida en esa pasada

    def predict_proba(self, p, X, batch_size=4096):
        p = pnp.array(p, requires_grad=False)
        return np.concatenate([np.asarray(self._prob_y_z(p, np.asarray(X[i:i + batch_size], float))[0])
                               for i in range(0, len(X), batch_size)])


ARQ = {g: Arquitectura(g[0], PERDIDAS[g[1]][1]) for g in GRUPOS}


# ── Simulación independiente (matrices 2×2, sin PennyLane) para verificar el circuito de 8 variables ─────────
def _rz(psi, t):
    return np.stack([psi[:, 0] * np.exp(-0.5j * t), psi[:, 1] * np.exp(0.5j * t)], axis=1)


def _ry(psi, t):
    c, s = np.cos(t / 2), np.sin(t / 2)
    return np.stack([c * psi[:, 0] - s * psi[:, 1], s * psi[:, 0] + c * psi[:, 1]], axis=1)


def perdida_numpy(arq, loss, p, X, y):
    w, b = p[arq.iw], p[arq.ib]
    psi = np.zeros((len(X), 2), complex)
    psi[:, 0] = 1
    for l in range(arq.n_capas):
        a = w[l] * X + b[l]
        for t in (0, 3):                                    # Rot(φ, θ, ω) = RZ(ω) · RY(θ) · RZ(φ)
            psi = _rz(_ry(_rz(psi, a[:, t]), a[:, t + 1]), a[:, t + 2])
        psi = _ry(psi, a[:, 6])
        if arq.n_vars == 8:
            psi = _rz(psi, a[:, 7])
    z = np.conj(psi[:, 0]) * psi[:, 1]
    ex, ey, ez = 2 * z.real, 2 * z.imag, np.abs(psi[:, 0]) ** 2 - np.abs(psi[:, 1]) ** 2
    if arq.readout == "xyz":
        r = arq.readout_slice.start
        prob = 1 / (1 + np.exp(-(p[r] * ex + p[r + 1] * ey + p[r + 2] * ez + p[r + 3])))
    else:
        prob = 0.5 * (1 - ez)
    F = 0.5 * (1 + (1 - 2 * y) * ez)
    if loss == "bce":
        v = np.mean(-(y * np.log(prob + 1e-9) + (1 - y) * np.log(1 - prob + 1e-9)))
    else:
        v = np.mean(1 - F) if loss == "infidelity" else np.mean((1 - F) ** 2)
    return v, prob


# ── Verificación ─────────────────────────────────────────────────────────────
rng = np.random.default_rng(7)
filas = []
for g in GRUPOS:
    n, l = g
    loss, arq = PERDIDAS[l][0], ARQ[g]
    pl = ModeloPL(arq, loss)
    Xb, yb = DATOS[g]["Xtr"][:64], DATOS[g]["ytr"][:64].astype(float)
    p = 0.5 * rng.normal(size=arq.n_params)
    v1, g1 = pl.loss_and_grad(p, Xb, yb)
    p1 = pl.predict_proba(p, Xb)
    if n == 7:
        ref = DRUModel(arq.circ7, loss=loss)
        v0, g0 = ref.loss_and_grad(p, Xb, yb)
        p0, contra, tol = ref.predict_proba(p, Xb), "motor NumPy del repo", 1e-5
    else:
        v0, p0 = perdida_numpy(arq, loss, p, Xb, yb)
        h = 1e-6
        g0 = np.array([(perdida_numpy(arq, loss, p + h * e, Xb, yb)[0] - perdida_numpy(arq, loss, p - h * e, Xb, yb)[0])
                       / (2 * h) for e in np.eye(arq.n_params)])
        contra, tol = "matrices 2×2 + diferencias finitas", 1e-6
    fila = dict(grupo=nombre(g), parametros=arq.n_params, comparado_con=contra, dif_prob=np.max(np.abs(p0 - p1)),
                dif_perdida=abs(v0 - v1), dif_grad_relativa=np.max(np.abs(g0 - g1)) / np.max(np.abs(g0)))
    filas.append(fila)
    assert fila["dif_prob"] < 1e-10 and fila["dif_perdida"] < 1e-10 and fila["dif_grad_relativa"] < tol, fila
display(pd.DataFrame(filas).style.format({"dif_prob": "{:.1e}", "dif_perdida": "{:.1e}", "dif_grad_relativa": "{:.1e}"})
        .hide(axis="index"))

# Con la RZ extra apagada, 8 variables ≡ 7 variables
for l in PERDIDAS:
    a7, a8 = ARQ[(7, l)], ARQ[(8, l)]
    p8 = 0.5 * rng.normal(size=a8.n_params)
    p8[a8.iw[:, 7]] = p8[a8.ib[:, 7]] = 0.0
    X8 = rng.uniform(-np.pi, np.pi, size=(64, 8))
    d = np.max(np.abs(ModeloPL(a8, PERDIDAS[l][0]).predict_proba(p8, X8)
                      - ModeloPL(a7, PERDIDAS[l][0]).predict_proba(p8[: a7.n_params], X8[:, :7])))
    assert d < 1e-12, (l, d)

# En el arranque capa por capa, cada parámetro de compuerta se entrena en alguna de las máscaras
for g in GRUPOS:
    arq = ARQ[g]
    vistos = np.zeros(arq.n_params)
    for capa in range(arq.n_capas):
        m = arq.mascara(capa)
        vistos[m.astype(bool) if m.shape == (arq.n_params,) else m] += 1
    assert np.all(vistos[np.concatenate([arq.iw.ravel(), arq.ib.ravel()])] >= 1), nombre(g)
print("Circuito verificado: 7 variables ≡ dru_core, 8 variables ≡ simulación independiente, 8 con RZ apagada ≡ 7.")

PennyLane 0.45.1


grupo,parametros,comparado_con,dif_prob,dif_perdida,dif_grad_relativa
7 variables · BCE,74,motor NumPy del repo,3.3e-16,0.0e+00,3.3e-09
7 variables · infidelidad lineal,70,motor NumPy del repo,5.6e-16,5.6e-17,7.3e-16
7 variables · infidelidad cuadrática,70,motor NumPy del repo,5.0e-16,0.0e+00,1.0e-15
8 variables · BCE,84,matrices 2×2 + diferencias finitas,2.2e-16,1.1e-16,8.1e-10
8 variables · infidelidad lineal,80,matrices 2×2 + diferencias finitas,8.9e-16,2.2e-16,7.1e-10
8 variables · infidelidad cuadrática,80,matrices 2×2 + diferencias finitas,8.9e-16,1.7e-16,4.9e-10


Circuito verificado: 7 variables ≡ dru_core, 8 variables ≡ simulación independiente, 8 con RZ apagada ≡ 7.


## 5 · Entrenamiento

`entrenar` es el mismo bucle del notebook de PennyLane (copia de `corrida_final.train`): una época por capa, dos fases con
Adam nuevo en cada una, early stopping por AUC de validación que corta solo la fase en curso, y se queda con los
parámetros de la mejor época.

**Guardado y reanudación.** Cada corrida se guarda en `corridas/<id>.npz` en cuanto termina (parámetros de la mejor época,
predicciones de validación, historial por época y un resumen). Al volver a ejecutar, las corridas ya guardadas se saltan.
Cada archivo lleva la receta y una huella de los datos; si no coinciden con la configuración actual, la celda se detiene
para no mezclar experimentos. `configuracion.json` guarda el diseño y las versiones de la primera sesión.

En modo completo, la primera corrida es la receta de la corrida final con la semilla 101. Al terminar se compara con lo
que dio en el notebook de PennyLane (mejor época 27, AUC de validación 0.8672): si coincide, la base está intacta.

In [ ]:
import time, zipfile, platform
from datetime import datetime
import sklearn
from sklearn.metrics import roc_auc_score

DEVICE = "default.qubit"
REFERENCIA = dict(mejor_epoca=27, auc_val=0.8672)   # receta de la corrida final, semilla 101 (notebook de PennyLane)


def entrenar(hp, arq, seed, data, verbose=False):
    """Copia de corrida_final.train con ModeloPL; la inicialización y las máscaras salen de `arq`."""
    Xtr, ytr, Xva, yva, use_val = data["Xtr"], data["ytr"], data["Xva"], data["yva"], data["use_val"]
    rng = np.random.default_rng(seed)
    model = ModeloPL(arq, loss=hp.loss, alpha=hp.hybrid_alpha, device=DEVICE)
    p = arq.init(hp, seed)

    def batches():
        o = rng.permutation(len(Xtr))
        for s in range(0, len(o), hp.batch_size):
            yield Xtr[o[s:s + hp.batch_size]], ytr[o[s:s + hp.batch_size]]

    for layer in range(hp.n_layers):                      # capa por capa
        opt, m = Adam(arq.n_params, lr=hp.layerwise_lr), arq.mascara(layer)
        for _ in range(hp.layerwise_epochs):
            for Xb, yb in batches():
                p = opt.step(p, model.loss_and_grad(p, Xb, yb)[1], m)

    best = dict(auc=-np.inf, p=p.copy(), epoch=0)
    hist, ep, stalled = [], 0, 0
    for s_i, (n_ep, lr) in enumerate(hp.stages):           # fases (Adam nuevo en cada una)
        opt = Adam(arq.n_params); opt.lr = lr
        stalled = 0 if hp.patience_per_stage else stalled
        for _ in range(n_ep):
            ep += 1
            losses = []
            for Xb, yb in batches():
                v, g = model.loss_and_grad(p, Xb, yb); p = opt.step(p, g); losses.append(v)
            row = dict(epoch=ep, stage=s_i, lr=lr, loss=float(np.mean(losses)))
            stop_stage = False
            if use_val:
                auc = roc_auc_score(yva, model.predict_proba(p, Xva)); row["val_auc"] = auc
                if auc > best["auc"] + hp.min_delta:
                    best, stalled = dict(auc=auc, p=p.copy(), epoch=ep), 0
                else:
                    stalled += 1
                    stop_stage = stalled >= hp.patience
            hist.append(row)
            if verbose:
                print(f"  época {ep:3d} fase {s_i} pérdida {row['loss']:.4f} AUC val {row.get('val_auc', float('nan')):.4f}")
            if stop_stage:
                break
    if not use_val:
        best = dict(auc=np.nan, p=p.copy(), epoch=ep)
    return model, best, hist


# ── Guardar y cargar corridas ────────────────────────────────────────────────
def archivo(c):
    return SALIDA / "corridas" / f"{run_id(c)}.npz"


def guardar_corrida(c, hp, model, best, hist, segundos):
    g = (c["n_vars"], c["perdida"])
    meta = dict(c, run_id=run_id(c), grupo=nombre(g), hp=repr(hp), huella_datos=HUELLA[g], modo=MODO,
                mejor_epoca=int(best["epoch"]), auc_val=float(best["auc"]), epocas=len(hist),
                segundos=round(segundos, 1), pennylane=qml.__version__, fecha=datetime.now().isoformat(timespec="seconds"))
    f = archivo(c)
    tmp = f.with_name(f.stem + ".tmp.npz")
    np.savez(tmp, meta=json.dumps(meta, ensure_ascii=False), p=best["p"],
             val_pred=model.predict_proba(best["p"], DATOS[g]["Xva"]),
             **{f"hist_{k}": np.array([h.get(k, np.nan) for h in hist], float)
                for k in ("epoch", "stage", "lr", "loss", "val_auc")})
    os.replace(tmp, f)                                     # el archivo aparece completo o no aparece
    return meta


def cargar_corrida(c):
    """Lo guardado de una corrida, o None si todavía no está. Comprueba que sea de esta misma configuración."""
    f = archivo(c)
    if not f.exists():
        return None
    try:
        with np.load(f, allow_pickle=False) as z:
            out = {k: z[k] for k in z.files}
        meta = json.loads(str(out.pop("meta")))
    except (OSError, ValueError, EOFError, KeyError, zipfile.BadZipFile) as e:
        print(f"  {f.name} está dañado ({type(e).__name__}); se vuelve a entrenar")
        return None
    if meta["hp"] != repr(hp_de(c)) or meta["huella_datos"] != HUELLA[(c["n_vars"], c["perdida"])]:
        raise RuntimeError(f"{f.name} es de otra configuración o de otros datos. Usa otro EXPERIMENTO en el paso 0 "
                           "para no mezclar resultados.")
    out["meta"] = meta
    return out


def comprobar_referencia(meta):
    ok = meta["mejor_epoca"] == REFERENCIA["mejor_epoca"] and abs(meta["auc_val"] - REFERENCIA["auc_val"]) < 5e-5
    print(("✔ " if ok else "⚠ ") + f"Receta de la corrida final, semilla 101: mejor época {meta['mejor_epoca']}, "
          f"AUC val {meta['auc_val']:.4f} (esperado {REFERENCIA['mejor_epoca']} y {REFERENCIA['auc_val']:.4f})"
          + ("" if ok else ". No coincide: revisa la versión de PennyLane y del repo antes de seguir."))


# ── Configuración del experimento: se guarda la primera vez y se compara al reanudar ──────────────────────
CONFIG = json.loads(json.dumps(dict(
    modo=MODO, etas=ETAS, lotes=LOTES, semillas=SEMILLAS, variables={str(k): v for k, v in VARIABLES.items()},
    perdidas={k: v[:2] for k, v in PERDIDAS.items()}, n_train=N_TRAIN, n_val=N_VAL, epocas_fase=EPOCAS_FASE,
    hp_base=repr(HP_BASE), huellas_datos={nombre(g): HUELLA[g] for g in GRUPOS})))
ENTORNO = dict(commit=COMMIT, python=platform.python_version(), numpy=np.__version__, sklearn=sklearn.__version__,
               pennylane=qml.__version__)
f_cfg = SALIDA / "configuracion.json"
if f_cfg.exists():
    previo = json.loads(f_cfg.read_text(encoding="utf-8"))
    if previo["experimento"] != CONFIG:
        distintos = [k for k in CONFIG if previo["experimento"].get(k) != CONFIG[k]]
        raise RuntimeError(f"{f_cfg} es de otro diseño de experimento (cambió: {', '.join(distintos)}). "
                           "Usa otro EXPERIMENTO en el paso 0 para no mezclar resultados.")
    cambios = {k: (previo["entorno"].get(k), v) for k, v in ENTORNO.items() if previo["entorno"].get(k) != v}
    if cambios:
        print("Aviso: el entorno cambió desde la primera sesión (antes → ahora):", cambios)
else:
    f_cfg.write_text(json.dumps(dict(experimento=CONFIG, entorno=ENTORNO,
                                     creado=datetime.now().isoformat(timespec="seconds")), indent=2, ensure_ascii=False),
                     encoding="utf-8")
print("configuración:", f_cfg)

configuración: /content/drive/MyDrive/SUSY_DRU/7v8_pennylane_v1/configuracion.json


In [ ]:
hechas = {run_id(c) for c in CORRIDAS if cargar_corrida(c) is not None}
pendientes = [c for c in CORRIDAS if run_id(c) not in hechas]
print(f"{len(hechas)} de {len(CORRIDAS)} corridas ya estaban guardadas; faltan {len(pendientes)}.")
REF101 = next(c for c in CORRIDAS if es_ref(c) and c["semilla"] == 101)
if MODO == "completo" and run_id(REF101) in hechas:
    comprobar_referencia(cargar_corrida(REF101)["meta"])

duraciones = []
for k, c in enumerate(pendientes, 1):
    g = (c["n_vars"], c["perdida"])
    hp = hp_de(c)
    t0 = time.perf_counter()
    model, best, hist = entrenar(hp, ARQ[g], c["semilla"], DATOS[g])
    seg = time.perf_counter() - t0
    meta = guardar_corrida(c, hp, model, best, hist, seg)
    duraciones.append(seg)
    resto = round(np.mean(duraciones) * (len(pendientes) - k) / 60)
    print(f"[{k:3d}/{len(pendientes)}] {run_id(c):42s} mejor época {best['epoch']:2d} · AUC val {best['auc']:.4f}"
          f" · {seg:4.0f} s · faltan ~{f'{resto // 60} h {resto % 60} min' if resto >= 60 else f'{resto} min'}")
    if MODO == "completo" and c == REF101:
        comprobar_referencia(meta)
print("Todas las corridas están guardadas en", SALIDA / "corridas")

0 de 108 corridas ya estaban guardadas; faltan 108.
[  1/108] v7_bce_eta0.01_b128_s101                   mejor época 27 · AUC val 0.8672 ·   63 s · faltan ~1 h 53 min
✔ Receta de la corrida final, semilla 101: mejor época 27, AUC val 0.8672 (esperado 27 y 0.8672)
[  2/108] v7_bce_eta0.01_b128_s202                   mejor época 18 · AUC val 0.8597 ·   42 s · faltan ~1 h 33 min
[  3/108] v7_bce_eta0.01_b128_s303                   mejor época 29 · AUC val 0.8599 ·   49 s · faltan ~1 h 30 min
[  4/108] v7_bce_eta0.005_b64_s101                   mejor época 25 · AUC val 0.8656 ·   93 s · faltan ~1 h 47 min
[  5/108] v7_bce_eta0.005_b64_s202                   mejor época 10 · AUC val 0.8595 ·   63 s · faltan ~1 h 46 min
[  6/108] v7_bce_eta0.005_b64_s303                   mejor época 30 · AUC val 0.8647 ·   94 s · faltan ~1 h 55 min
[  7/108] v7_bce_eta0.005_b128_s101                  mejor época 20 · AUC val 0.8591 ·   45 s · faltan ~1 h 48 min
[  8/108] v7_bce_eta0.005_b128_s202           

## 6 · Selección con validación

Para cada grupo se promedia el AUC de validación de las semillas de cada ajuste (η, lote) y gana el mayor
(empate exacto: menor η y luego menor lote). La selección se guarda en `tablas/seleccion.json` antes de mirar test; si
ya existía una distinta, la celda se detiene para no volver a elegir después de ver test.

In [ ]:
filas, faltan = [], []
for c in CORRIDAS:
    z = cargar_corrida(c)
    if z is None:
        faltan.append(run_id(c))
        continue
    m = z["meta"]
    filas.append(dict(grupo=nombre((c["n_vars"], c["perdida"])), **c, auc_val=m["auc_val"], mejor_epoca=m["mejor_epoca"],
                      epocas=m["epocas"], segundos=m["segundos"]))
if faltan:
    raise RuntimeError(f"Faltan {len(faltan)} corridas (por ejemplo {faltan[0]}). Vuelve a ejecutar el paso 5; "
                       "el test no se toca hasta tenerlas todas.")
VAL = pd.DataFrame(filas)
VAL["orden"] = [GRUPOS.index((n, l)) for n, l in zip(VAL.n_vars, VAL.perdida)]

CAND = (VAL.groupby(["orden", "grupo", "n_vars", "perdida", "eta", "lote"], as_index=False)
           .agg(auc_val_media=("auc_val", "mean"), auc_val_sd=("auc_val", "std"), semillas=("semilla", "count"),
                mejor_epoca_media=("mejor_epoca", "mean")))
SEL = (CAND.sort_values(["orden", "auc_val_media", "eta", "lote"], ascending=[True, False, True, True])
           .drop_duplicates("orden").reset_index(drop=True))

seleccion = json.loads(json.dumps(SEL[["grupo", "n_vars", "perdida", "eta", "lote", "auc_val_media"]].to_dict("records"),
                                  default=lambda o: o.item()))
f_sel = SALIDA / "tablas" / "seleccion.json"
if f_sel.exists() and json.loads(f_sel.read_text(encoding="utf-8")) != seleccion:
    raise RuntimeError(f"{f_sel} ya tiene otra selección. No se vuelve a elegir después de haber mirado test.")
f_sel.write_text(json.dumps(seleccion, indent=2, ensure_ascii=False), encoding="utf-8")
VAL.drop(columns="orden").to_csv(SALIDA / "tablas" / "validacion_por_corrida.csv", index=False)
CAND.drop(columns="orden").to_csv(SALIDA / "tablas" / "validacion_por_ajuste.csv", index=False)

print("AUC de validación medio por ajuste (filas: η · columnas: lote)")
display(CAND.pivot_table(index=["grupo", "eta"], columns="lote", values="auc_val_media", sort=False).style.format("{:.4f}"))
print("Ajuste elegido en cada grupo")
display(SEL[["grupo", "eta", "lote", "auc_val_media", "auc_val_sd", "mejor_epoca_media"]]
        .style.format({"auc_val_media": "{:.4f}", "auc_val_sd": "{:.4f}", "mejor_epoca_media": "{:.1f}"}).hide(axis="index"))

AUC de validación medio por ajuste (filas: η · columnas: lote)


Ajuste elegido en cada grupo


grupo,eta,lote,auc_val_media,auc_val_sd,mejor_epoca_media
7 variables · BCE,0.010000,64,0.8656,0.0063,24.3
7 variables · infidelidad lineal,0.010000,128,0.8425,0.0035,12.0
7 variables · infidelidad cuadrática,0.005000,64,0.8644,0.0034,19.3
8 variables · BCE,0.005000,128,0.8508,0.0057,24.0
8 variables · infidelidad lineal,0.010000,64,0.8343,0.0047,11.0
8 variables · infidelidad cuadrática,0.010000,64,0.8535,0.0014,19.3


## 7 · Evaluación en test

Los modelos elegidos (un ajuste por grupo, todas sus semillas) se evalúan una sola vez en los 5000 eventos de test.

- **Por semilla:** AUC, AP (precisión media), accuracy, precisión, recall, F1 y Brier, con umbral p ≥ 0.5. Se resume como
  media ± desviación estándar entre semillas.
- **Ensemble:** el promedio de las probabilidades de las semillas, con intervalo de confianza del 95 % por bootstrap
  (2000 remuestreos de los eventos de test).
- **8 − 7 variables:** diferencia de AUC del ensemble con la misma pérdida. Es pareada: los mismos remuestreos para los
  dos grupos. Si el intervalo no incluye el 0, la diferencia no se debe solo a qué eventos tocaron en test. No mide la
  variación entre semillas: esa se ve en la desviación estándar.

In [ ]:
from sklearn.metrics import (accuracy_score, average_precision_score, brier_score_loss, f1_score,
                             precision_score, recall_score)

METRICAS = {"auc": "AUC", "ap": "AP (precisión media)", "accuracy": "accuracy", "precision": "precisión",
            "recall": "recall", "f1": "F1", "brier": "Brier"}


def metricas(y, prob, umbral=0.5):
    pred = (prob >= umbral).astype(int)
    return dict(auc=roc_auc_score(y, prob), ap=average_precision_score(y, prob), accuracy=accuracy_score(y, pred),
                precision=precision_score(y, pred, zero_division=0), recall=recall_score(y, pred, zero_division=0),
                f1=f1_score(y, pred, zero_division=0), brier=brier_score_loss(y, prob))


PRED_TEST, filas = {}, []
for s in SEL.itertuples():
    g = (int(s.n_vars), s.perdida)
    model = ModeloPL(ARQ[g], PERDIDAS[g[1]][0])
    P = []
    for semilla in SEMILLAS:
        c = dict(n_vars=g[0], perdida=g[1], eta=float(s.eta), lote=int(s.lote), semilla=semilla)
        z = cargar_corrida(c)
        prob = model.predict_proba(z["p"], DATOS[g]["Xte"])
        P.append(prob)
        filas.append(dict(grupo=nombre(g), n_vars=g[0], perdida=g[1], eta=c["eta"], lote=c["lote"], semilla=semilla,
                          mejor_epoca=z["meta"]["mejor_epoca"], **metricas(YTE, prob)))
    PRED_TEST[g] = np.array(P)
    np.savez(SALIDA / "test" / f"v{g[0]}_{g[1]}.npz", y=YTE, prob=PRED_TEST[g], semillas=np.array(SEMILLAS),
             eta=c["eta"], lote=c["lote"])
POR_SEMILLA = pd.DataFrame(filas)
ENS = {g: P.mean(axis=0) for g, P in PRED_TEST.items()}

# Bootstrap (los mismos remuestreos para todos los grupos → comparaciones pareadas)
N_BOOT = 2000 if MODO == "completo" else 200
BOOT = np.random.default_rng(2026).integers(0, len(YTE), size=(N_BOOT, len(YTE)))
AUC_BOOT = {g: np.array([roc_auc_score(YTE[i], ENS[g][i]) for i in BOOT]) for g in ENS}

filas = []
for s in SEL.itertuples():
    g = (int(s.n_vars), s.perdida)
    d = POR_SEMILLA[POR_SEMILLA.grupo == nombre(g)]
    fila = dict(grupo=nombre(g), n_vars=g[0], perdida=g[1], eta=s.eta, lote=s.lote, parametros=ARQ[g].n_params,
                semillas=len(d))
    for k in METRICAS:
        fila[f"{k}_media"], fila[f"{k}_sd"] = d[k].mean(), d[k].std(ddof=1)
    fila["auc_ensemble"] = roc_auc_score(YTE, ENS[g])
    fila["auc_ensemble_ic95_lo"], fila["auc_ensemble_ic95_hi"] = np.percentile(AUC_BOOT[g], [2.5, 97.5])
    filas.append(fila)
RESUMEN = pd.DataFrame(filas)

DIFS = pd.DataFrame([dict(perdida=PERDIDAS[l][2],
                          dif_auc_ensemble=RESUMEN.set_index("grupo").auc_ensemble[nombre((8, l))]
                          - RESUMEN.set_index("grupo").auc_ensemble[nombre((7, l))],
                          ic95_lo=np.percentile(AUC_BOOT[(8, l)] - AUC_BOOT[(7, l)], 2.5),
                          ic95_hi=np.percentile(AUC_BOOT[(8, l)] - AUC_BOOT[(7, l)], 97.5),
                          dif_auc_media_semillas=RESUMEN.set_index("grupo").auc_media[nombre((8, l))]
                          - RESUMEN.set_index("grupo").auc_media[nombre((7, l))])
                     for l in PERDIDAS])

POR_SEMILLA.to_csv(SALIDA / "tablas" / "test_por_semilla.csv", index=False)
RESUMEN.to_csv(SALIDA / "tablas" / "test_resumen.csv", index=False)
DIFS.to_csv(SALIDA / "tablas" / "test_diferencia_8_menos_7.csv", index=False)

if MODO == "rapido":
    print("PRUEBA RÁPIDA: modelos entrenados con muy pocos datos y épocas; estos números no significan nada.\n")
tabla = pd.DataFrame({"grupo": RESUMEN.grupo, "η · lote": [f"{e:g} · {b}" for e, b in zip(RESUMEN.eta, RESUMEN.lote)]})
for k, t in METRICAS.items():
    tabla[t] = [f"{m:.4f} ± {sd:.4f}" for m, sd in zip(RESUMEN[f"{k}_media"], RESUMEN[f"{k}_sd"])]
tabla["AUC ensemble [IC 95 %]"] = [f"{a:.4f} [{lo:.4f}, {hi:.4f}]" for a, lo, hi in
                                   zip(RESUMEN.auc_ensemble, RESUMEN.auc_ensemble_ic95_lo, RESUMEN.auc_ensemble_ic95_hi)]
print(f"Test · media ± desviación estándar entre {len(SEMILLAS)} semillas")
display(tabla.style.hide(axis="index"))
print("Diferencia de AUC del ensemble, 8 − 7 variables (IC 95 % bootstrap pareado)")
display(DIFS.style.format({c: "{:+.4f}" for c in DIFS.columns if c != "perdida"}).hide(axis="index"))

Test · media ± desviación estándar entre 3 semillas


grupo,η · lote,AUC,AP (precisión media),accuracy,precisión,recall,F1,Brier,AUC ensemble [IC 95 %]
7 variables · BCE,0.01 · 64,0.8486 ± 0.0030,0.8664 ± 0.0036,0.7721 ± 0.0038,0.7972 ± 0.0354,0.7337 ± 0.0437,0.7628 ± 0.0085,0.1576 ± 0.0006,"0.8529 [0.8428, 0.8635]"
7 variables · infidelidad lineal,0.01 · 128,0.8317 ± 0.0019,0.8332 ± 0.0054,0.7651 ± 0.0052,0.7889 ± 0.0125,0.7244 ± 0.0221,0.7550 ± 0.0085,0.1812 ± 0.0006,"0.8361 [0.8254, 0.8476]"
7 variables · infidelidad cuadrática,0.005 · 64,0.8505 ± 0.0045,0.8661 ± 0.0061,0.7741 ± 0.0063,0.7979 ± 0.0347,0.7377 ± 0.0406,0.7654 ± 0.0054,0.1570 ± 0.0028,"0.8554 [0.8453, 0.8661]"
8 variables · BCE,0.005 · 128,0.8410 ± 0.0038,0.8570 ± 0.0039,0.7656 ± 0.0037,0.7685 ± 0.0194,0.7617 ± 0.0319,0.7645 ± 0.0079,0.1623 ± 0.0020,"0.8454 [0.8350, 0.8563]"
8 variables · infidelidad lineal,0.01 · 64,0.8241 ± 0.0023,0.8310 ± 0.0032,0.7613 ± 0.0015,0.7786 ± 0.0092,0.7304 ± 0.0118,0.7536 ± 0.0023,0.1922 ± 0.0041,"0.8276 [0.8162, 0.8392]"
8 variables · infidelidad cuadrática,0.01 · 64,0.8434 ± 0.0026,0.8613 ± 0.0017,0.7684 ± 0.0035,0.7846 ± 0.0306,0.7432 ± 0.0408,0.7622 ± 0.0074,0.1607 ± 0.0017,"0.8480 [0.8376, 0.8591]"


Diferencia de AUC del ensemble, 8 − 7 variables (IC 95 % bootstrap pareado)


perdida,dif_auc_ensemble,ic95_lo,ic95_hi,dif_auc_media_semillas
BCE,-0.0075,-0.0115,-0.0035,-0.0076
infidelidad lineal,-0.0085,-0.0128,-0.0042,-0.0077
infidelidad cuadrática,-0.0074,-0.0109,-0.0040,-0.0070


## 8 · Gráficas

Todas se guardan en `figuras/` (PNG). Azul = 7 variables, naranja = 8 variables, en todas las gráficas.

1. `metricas_test.png`: las siete métricas de test por grupo (media ± desviación estándar entre semillas).
2. `diferencia_auc_8_menos_7.png`: diferencia de AUC del ensemble con su intervalo del 95 %.
3. `roc_test.png` y `precision_recall_test.png`: curvas del ensemble de cada grupo.
4. `busqueda_validacion.png`: AUC de validación medio de cada ajuste de la búsqueda, con el elegido marcado.
5. `curvas_validacion.png`: AUC de validación por época de los ajustes elegidos (una línea por semilla).
6. `distribucion_scores.png`: probabilidad de señal que da el ensemble, separada por clase real.
7. `calibracion.png`: probabilidad predicha frente a la fracción real de señal.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
from matplotlib.ticker import MaxNLocator
from sklearn.metrics import roc_curve, precision_recall_curve
from sklearn.calibration import calibration_curve

TINTA, TINTA2, TENUE, REJILLA, EJE, FONDO = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7", "#fcfcfb"
COLOR_VARS = {7: "#2a78d6", 8: "#eb6834"}            # azul / naranja (par validado para daltonismo)
COLOR_CLASE = {0: "#1baf7a", 1: "#6250d6"}            # fondo / señal
AZULES = LinearSegmentedColormap.from_list("azules", ["#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"])
plt.rcParams.update({
    "figure.facecolor": FONDO, "axes.facecolor": FONDO, "savefig.facecolor": FONDO,
    "axes.edgecolor": EJE, "axes.linewidth": 0.8, "axes.labelcolor": TINTA2, "axes.titlecolor": TINTA,
    "axes.titlesize": 10.5, "axes.labelsize": 9.5, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": REJILLA, "grid.linewidth": 0.8, "grid.linestyle": "-", "axes.axisbelow": True,
    "xtick.color": TENUE, "ytick.color": TENUE, "xtick.labelcolor": TINTA2, "ytick.labelcolor": TINTA2,
    "xtick.labelsize": 8.5, "ytick.labelsize": 8.5, "legend.frameon": False, "legend.fontsize": 8.5,
    "text.color": TINTA, "font.size": 9.5, "lines.linewidth": 1.6, "lines.solid_capstyle": "round",
})
AVISO = "\nPRUEBA RÁPIDA: los números no significan nada" if MODO == "rapido" else ""
NOMBRES_L = {l: PERDIDAS[l][2] for l in PERDIDAS}
leyenda_vars = [Line2D([], [], color=COLOR_VARS[n], marker="o", ms=7, mec=FONDO, mew=1.5, lw=0, label=f"{n} variables")
                for n in VARIABLES]


def guardar(fig, archivo_png):
    fig.savefig(SALIDA / "figuras" / archivo_png, dpi=150, bbox_inches="tight")
    plt.show()


# 1 · Métricas de test (puntos con barra de ± 1 desviación estándar entre semillas)
fig, axes = plt.subplots(2, 4, figsize=(13, 5.8), sharey=True)
for ax, (k, titulo) in zip(axes.flat, METRICAS.items()):
    for n, dy in ((7, -0.14), (8, 0.14)):
        for i, l in enumerate(PERDIDAS):
            d = POR_SEMILLA[(POR_SEMILLA.n_vars == n) & (POR_SEMILLA.perdida == l)][k]
            ax.errorbar(d.mean(), i + dy, xerr=d.std(ddof=1), fmt="o", color=COLOR_VARS[n], ms=7, mec=FONDO, mew=1.5,
                        elinewidth=1.6, capsize=0)
    ax.set_title(titulo + (" (menor es mejor)" if k == "brier" else ""), loc="left")
    ax.grid(axis="y", visible=False)
    ax.set_yticks(range(len(PERDIDAS)), list(NOMBRES_L.values()))
    ax.margins(x=0.25)
axes[0, 0].invert_yaxis()
ax = axes.flat[-1]
ax.axis("off")
ax.legend(handles=leyenda_vars, loc="upper left", fontsize=9.5)
ax.text(0.02, 0.45, f"punto: media de {len(SEMILLAS)} semillas\nbarra: ± 1 desviación estándar\numbral p ≥ 0.5 para "
        "accuracy,\nprecisión, recall y F1", transform=ax.transAxes, va="top", color=TINTA2, fontsize=9)
fig.suptitle("Test · modelos elegidos con validación" + AVISO, x=0.01, ha="left", fontsize=12)
fig.tight_layout()
guardar(fig, "metricas_test.png")

# 2 · Diferencia de AUC 8 − 7 (ensemble, IC 95 % bootstrap pareado)
fig, ax = plt.subplots(figsize=(7.5, 2.6))
for i, r in enumerate(DIFS.itertuples()):
    ax.errorbar(r.dif_auc_ensemble, i, xerr=[[r.dif_auc_ensemble - r.ic95_lo], [r.ic95_hi - r.dif_auc_ensemble]],
                fmt="o", color=TINTA, ms=7, mec=FONDO, mew=1.5, elinewidth=1.6, capsize=0)
    ax.annotate(f"{r.dif_auc_ensemble:+.4f}  [{r.ic95_lo:+.4f}, {r.ic95_hi:+.4f}]", (r.ic95_hi, i),
                xytext=(8, 0), textcoords="offset points", va="center", color=TINTA2, fontsize=8.5)
ax.axvline(0, color=TENUE, lw=1.2)
ax.set_yticks(range(len(DIFS)), DIFS.perdida)
ax.invert_yaxis()
ax.grid(axis="y", visible=False)
ax.margins(x=0.6)
ax.set_xlabel("AUC(8 variables) − AUC(7 variables) · ensemble en test")
ax.set_title("¿Cambia el AUC al pasar a las 8 variables primarias?" + AVISO, loc="left")
fig.tight_layout()
guardar(fig, "diferencia_auc_8_menos_7.png")

# 3 · Curvas ROC y precisión-recall del ensemble
for tipo in ("roc", "pr"):
    fig, axes = plt.subplots(1, 3, figsize=(13, 4.3), sharey=True)
    for ax, l in zip(axes, PERDIDAS):
        for n in VARIABLES:
            y_s = ENS[(n, l)]
            if tipo == "roc":
                fpr, tpr, _ = roc_curve(YTE, y_s)
                ax.plot(fpr, tpr, color=COLOR_VARS[n], label=f"{n} variables · AUC {roc_auc_score(YTE, y_s):.4f}")
            else:
                prec, rec, _ = precision_recall_curve(YTE, y_s)
                ax.plot(rec, prec, color=COLOR_VARS[n], label=f"{n} variables · AP {average_precision_score(YTE, y_s):.4f}")
        if tipo == "roc":
            ax.plot([0, 1], [0, 1], color=EJE, lw=1)
            ax.set_xlabel("tasa de falsos positivos (fondo aceptado)")
        else:
            ax.axhline(YTE.mean(), color=EJE, lw=1)
            ax.set_xlabel("recall (señal recuperada)")
        ax.set_title(NOMBRES_L[l], loc="left")
        ax.set_xlim(0, 1); ax.set_ylim(0, 1.01)
        ax.legend(loc="lower right" if tipo == "roc" else "lower left")
    axes[0].set_ylabel("tasa de verdaderos positivos" if tipo == "roc" else "precisión")
    fig.suptitle(("Curvas ROC" if tipo == "roc" else "Curvas precisión-recall") + " del ensemble en test" + AVISO,
                 x=0.01, ha="left", fontsize=12)
    fig.tight_layout()
    guardar(fig, "roc_test.png" if tipo == "roc" else "precision_recall_test.png")

# 4 · Búsqueda: AUC de validación medio por (η, lote), con el ajuste elegido marcado
fig, axes = plt.subplots(len(VARIABLES), len(PERDIDAS), figsize=(11, 2.0 + 1.6 * len(VARIABLES)), squeeze=False)
for (ax, g) in zip(axes.flat, GRUPOS):
    tab = CAND[CAND.grupo == nombre(g)].pivot(index="eta", columns="lote", values="auc_val_media")
    vmin, vmax = np.nanmin(tab.values), np.nanmax(tab.values)          # escala de color propia de cada grupo
    ax.pcolormesh(tab.values, cmap=AZULES, vmin=vmin, vmax=vmax if vmax > vmin else vmin + 1e-9,
                  edgecolors=FONDO, linewidth=2)
    ax.invert_yaxis()
    elegido = SEL[SEL.grupo == nombre(g)].iloc[0]
    for i, eta in enumerate(tab.index):
        for j, lote in enumerate(tab.columns):
            v = tab.values[i, j]
            oscuro = (v - vmin) / max(vmax - vmin, 1e-12) > 0.55
            es_sel = eta == elegido.eta and lote == elegido.lote
            ax.text(j + 0.5, i + 0.5, f"{v:.4f}" + (" ★" if es_sel else ""), ha="center", va="center", fontsize=9,
                    color="#ffffff" if oscuro else TINTA, fontweight="bold" if es_sel else "normal")
    ax.set_xticks(np.arange(len(tab.columns)) + 0.5, [f"lote {b}" for b in tab.columns])
    ax.set_yticks(np.arange(len(tab.index)) + 0.5, [f"η = {e:g}" for e in tab.index])
    ax.grid(False)
    for s in ax.spines.values():
        s.set_visible(False)
    ax.tick_params(length=0)
    ax.set_title(nombre(g), loc="left")
fig.suptitle(f"Validación · AUC medio de {len(SEMILLAS)} semillas por ajuste (★ = elegido; "
             "el color compara solo dentro de cada grupo)" + AVISO,
             x=0.01, ha="left", fontsize=12)
fig.tight_layout()
guardar(fig, "busqueda_validacion.png")

# 5 · Curvas de validación por época de los ajustes elegidos
fig, axes = plt.subplots(len(VARIABLES), len(PERDIDAS), figsize=(13, 2.2 + 2.4 * len(VARIABLES)),
                         sharex=True, sharey=True, squeeze=False)
for ax, g in zip(axes.flat, GRUPOS):
    e = SEL[SEL.grupo == nombre(g)].iloc[0]
    for semilla in SEMILLAS:
        z = cargar_corrida(dict(n_vars=g[0], perdida=g[1], eta=float(e.eta), lote=int(e.lote), semilla=semilla))
        ax.plot(z["hist_epoch"], z["hist_val_auc"], color=COLOR_VARS[g[0]], lw=1.4, alpha=0.85)
        mejor = z["meta"]["mejor_epoca"]
        if mejor:
            ax.plot(mejor, z["hist_val_auc"][mejor - 1], "o", color=COLOR_VARS[g[0]], ms=7, mec=FONDO, mew=1.5)
    fin_fase1 = hp_de(dict(n_vars=g[0], perdida=g[1], eta=float(e.eta), lote=int(e.lote), semilla=0)).stages[0][0]
    ax.axvline(fin_fase1 + 0.5, color=EJE, lw=1)
    ax.set_title(f"{nombre(g)} · η = {e.eta:g}, lote {int(e.lote)}", loc="left")
for ax in axes[-1]:
    ax.set_xlabel("época (después del arranque capa por capa)")
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
for ax in axes[:, 0]:
    ax.set_ylabel("AUC de validación")
fig.suptitle(f"Validación por época · una línea por semilla, punto = mejor época, raya = fin de la fase 1" + AVISO,
             x=0.01, ha="left", fontsize=12)
fig.tight_layout()
guardar(fig, "curvas_validacion.png")

# 6 · Distribución de la probabilidad de señal del ensemble, por clase real
bins = np.linspace(0, 1, 41)
fig, axes = plt.subplots(len(VARIABLES), len(PERDIDAS), figsize=(13, 2.2 + 2.3 * len(VARIABLES)),
                         sharex=True, squeeze=False)
for ax, g in zip(axes.flat, GRUPOS):
    for clase, etiqueta in ((0, "fondo"), (1, "señal")):
        v = ENS[g][YTE == clase]
        ax.hist(v, bins=bins, color=COLOR_CLASE[clase], alpha=0.12)
        ax.hist(v, bins=bins, color=COLOR_CLASE[clase], histtype="step", lw=1.6, label=etiqueta)
    ax.axvline(0.5, color=EJE, lw=1)
    ax.set_title(nombre(g), loc="left")
    ax.grid(axis="x", visible=False)
axes[0, 0].legend(loc="upper center")
for ax in axes[-1]:
    ax.set_xlabel("probabilidad de señal (ensemble)")
for ax in axes[:, 0]:
    ax.set_ylabel("eventos de test")
fig.suptitle("Distribución de la probabilidad de señal por clase real (raya = umbral 0.5)" + AVISO,
             x=0.01, ha="left", fontsize=12)
fig.tight_layout()
guardar(fig, "distribucion_scores.png")

# 7 · Calibración
fig, axes = plt.subplots(1, 3, figsize=(13, 4.3), sharey=True)
for ax, l in zip(axes, PERDIDAS):
    for n in VARIABLES:
        frac, media = calibration_curve(YTE, ENS[(n, l)], n_bins=10, strategy="quantile")
        ax.plot(media, frac, "-o", color=COLOR_VARS[n], ms=6, mec=FONDO, mew=1.5,
                label=f"{n} variables · Brier {brier_score_loss(YTE, ENS[(n, l)]):.4f}")
    ax.plot([0, 1], [0, 1], color=EJE, lw=1)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_xlabel("probabilidad predicha (media por decil)")
    ax.set_title(NOMBRES_L[l], loc="left")
    ax.legend(loc="upper left")
axes[0].set_ylabel("fracción real de señal")
fig.suptitle("Calibración del ensemble en test (diagonal = calibración perfecta)" + AVISO, x=0.01, ha="left", fontsize=12)
fig.tight_layout()
guardar(fig, "calibracion.png")
print("Gráficas guardadas en", SALIDA / "figuras")

## 9 · Resumen y equivalencias con la corrida final

Las equivalencias con los nombres de la corrida final (M2, M3, …) se guardan en `tablas/equivalencias_corrida_final.csv`
para usarlas como leyenda en el README. En este notebook los grupos se llaman siempre por lo que son.

| Archivo o carpeta | Contenido |
|---|---|
| `configuracion.json` | diseño del experimento, huellas de los datos y versiones de la primera sesión |
| `corridas/` | una corrida por archivo: parámetros de la mejor época, predicciones de validación e historial |
| `tablas/validacion_*.csv`, `tablas/seleccion.json` | resultados de validación y ajuste elegido en cada grupo |
| `test/` | probabilidades de test de los modelos elegidos (semillas × eventos) |
| `tablas/test_*.csv` | métricas de test por semilla, resumen por grupo y diferencias 8 − 7 |
| `figuras/` | las gráficas del paso 8 |

**Notas para interpretar**
- Con 8 variables y lectura solo en ⟨Z⟩ (las dos infidelidades), la RZ de la última capa no cambia ⟨Z⟩: su peso y su
  sesgo no se entrenan (gradiente 0). Con BCE sí cuentan, porque la lectura usa también ⟨X⟩ y ⟨Y⟩.
- La primera RZ de la primera capa actúa sobre |0⟩ y solo cambia una fase global: sus dos parámetros no hacen nada.
  Pasa igual con 7 y con 8 variables, y en el repo.
- Cada ángulo φ por separado casi no distingue señal de fondo por la simetría azimutal; lo que distingue son sus
  diferencias. Que el circuito las aproveche es justo una de las cosas que mide la comparación.
- Pasar de 7 a 8 variables cambia también el número de compuertas y de parámetros (70 → 80 en el circuito).

In [ ]:
ref = pd.read_csv(REPO / "corrida_final" / "resultados_test.csv").set_index("modelo")
filas = []
for g in GRUPOS:
    hp = receta(*g, ETA_BASE, HP_BASE.batch_size)
    iguales = [k for k, v in cf.CONFIGS.items() if v == hp]
    fila = dict(grupo=nombre(g), eta=ETA_BASE, lote=HP_BASE.batch_size,
                configuracion_corrida_final=", ".join(iguales) if iguales else "— (nueva)")
    en_ref = [k for k in iguales if k in ref.index]
    if en_ref:
        r = ref.loc[en_ref[0]]
        fila.update(corrida_final_auc_ensemble=r.test_auc_ens, corrida_final_ic95=f"[{r.ic95_lo:.4f}, {r.ic95_hi:.4f}]",
                    corrida_final_auc_por_semilla=f"{r.test_auc_media:.4f} ± {r.test_auc_sd:.4f}")
    filas.append(fila)
EQUIV = pd.DataFrame(filas)
EQUIV.to_csv(SALIDA / "tablas" / "equivalencias_corrida_final.csv", index=False)
print("Leyenda para el README (receta con η = 0.01 y lote 128; la corrida final usó 5 semillas):")
display(EQUIV.style.hide(axis="index"))

print("\nArchivos en", SALIDA)
for sub in ("", "corridas", "test", "tablas", "figuras"):
    archivos = [f for f in (SALIDA / sub).iterdir() if f.is_file()]
    print(f"  {sub or '.':9s} {len(archivos):4d} archivos · {sum(f.stat().st_size for f in archivos) / 1e6:6.2f} MB")

Leyenda para el README (receta con η = 0.01 y lote 128; la corrida final usó 5 semillas):


grupo,eta,lote,configuracion_corrida_final,corrida_final_auc_ensemble,corrida_final_ic95,corrida_final_auc_por_semilla
7 variables · BCE,0.010000,128,M2,0.854853,"[0.8443, 0.8654]",0.8483 ± 0.0034
7 variables · infidelidad lineal,0.010000,128,M3-lin,0.835983,"[0.8242, 0.8475]",0.8321 ± 0.0017
7 variables · infidelidad cuadrática,0.010000,128,M3-cua,0.853526,"[0.8432, 0.8637]",0.8494 ± 0.0025
8 variables · BCE,0.010000,128,— (nueva),nan,nan,nan
8 variables · infidelidad lineal,0.010000,128,— (nueva),nan,nan,nan
8 variables · infidelidad cuadrática,0.010000,128,— (nueva),nan,nan,nan



Archivos en /content/drive/MyDrive/SUSY_DRU/7v8_pennylane_v1
  .            1 archivos ·   0.00 MB
  corridas   108 archivos ·   1.63 MB
  test         6 archivos ·   0.97 MB
  tablas       7 archivos ·   0.02 MB
  figuras      8 archivos ·   1.00 MB
